# QuantumFinance AI Agent — Demo end-to-end (Etapa 7)

Este notebook demonstra o sistema completo, do início ao fim:

1. Recomendação completa via **pipeline determinístico**
2. Perguntas pontuais via **roteamento dinâmico** (ReAct genuíno — o LLM decide quais tools chamar)
3. Pergunta comparativa entre dois tickers
4. Histórico de recomendações persistido em CSV
5. Resumo do backtest histórico (Etapas 6 e 6.5)

Cada célula faz chamadas reais: LLM via DeepInfra, preços via yfinance, notícias via RSS, sentimento via FinBERT-PT-BR. Pré-requisito: `.env` com `DEEPINFRA_API_KEY` configurado (ver `README.md`).

In [1]:
from langchain_core.messages import HumanMessage

from quantumfinance.agents.orchestrator import app, ask
from quantumfinance.output.storage import load_recommendations
from quantumfinance.universe import TICKERS

print(f"Tickers monitorados: {', '.join(TICKERS)}")

C:\Users\ASUS\Desktop\MBA\ENTREGAS\ENTREGA_MULTI-AGENTS\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Tickers monitorados: PETR4, VALE3, BBAS3, ITUB4


## 1. Recomendação completa — pipeline determinístico

O roteador identifica que a pergunta pede uma recomendação completa e segue o pipeline fixo: `pipeline_market → pipeline_sentiment → pipeline_decision`. As 3 tools são chamadas direto em Python — o LLM só entra no fim, para narrar o resultado já calculado. A recomendação fica registrada em `data/recommendations.csv`.

In [2]:
resposta = ask("Qual a recomendação para PETR4 hoje?")
print(resposta)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 30816.80it/s]

A recomendação atual para a ação **PETR4** é **AGUARDAR**.

A confiança nessa decisão é de **50%**, o que indica um cenário de incerteza, com sinais mistos no mercado.

### Principais fatores que influenciaram essa recomendação:

- **RSI (Índice de Força Relativa) em 28,93**: Esse valor indica que a ação está em **sobrevenda**, o que pode ser um sinal positivo, sugerindo que pode estar próximo de uma recuperação.
  
- **MACD com sinal negativo (bearish)**: Isso aponta para uma **tendência de queda** no momento, um fator que pressiona contra uma entrada agora.

- **Sentimento do mercado NEGATIVO (score: 0,0431)**: As notícias recentes sobre a Petrobras e o mercado em geral estão com viés negativo, influenciando o preço da ação.

Em resumo, há **dois sinais de alta** (como o RSI em sobrevenda) e **dois sinais de baixa** (MACD negativo e sentimento negativo), o que cria um impasse. Por isso, a melhor decisão no momento é **aguardar** para ver como os próximos movimentos se desenrolam ante

## 2. Pergunta pontual sobre mercado — roteamento dinâmico (ReAct genuíno)

Quando a pergunta não pede uma recomendação completa, o roteador manda direto para o `MarketAgent`, que decide por conta própria chamar a tool `get_market_features` — sem sequência pré-determinada, ReAct de verdade. A função abaixo mostra o rastro de tool calls junto com a resposta final, para tornar essa decisão visível.

In [3]:
def show_trace(question: str) -> None:
    """Mostra o rastro de tool calls e a resposta final para uma pergunta."""
    result = app.invoke({"messages": [HumanMessage(content=question)]})
    for message in result["messages"]:
        tool_calls = getattr(message, "tool_calls", None)
        if tool_calls:
            for call in tool_calls:
                print(f"[tool call] {call['name']}({call['args']})")
        elif type(message).__name__ == "ToolMessage":
            print(f"[tool result] {message.content}")
    print("\n--- Resposta final ---")
    print(result["messages"][-1].content)


show_trace("Qual o RSI e o MACD de VALE3 agora?")

[tool call] get_market_features({'ticker': 'VALE3'})
[tool result] {"close": 77.73, "volume": 22533600, "rsi": 40.52, "macd": -0.7909, "macd_signal_line": -0.7242, "macd_signal": "bearish", "sma_20": 80.54, "sma_50": 82.48, "ema_20": 80.37, "bollinger_upper": 84.8, "bollinger_lower": 76.27}

--- Resposta final ---
Os dados atuais para VALE3 são:

- Preço de fechamento: 77.73
- Volume: 22.533.600
- RSI: 40.52
- MACD: -0.7909
- Linha de sinal do MACD: -0.7242
- Sinal do MACD: bearish
- SMA 20: 80.54
- SMA 50: 82.48
- EMA 20: 80.37
- Banda de Bollinger superior: 84.8
- Banda de Bollinger inferior: 76.27


## 3. Pergunta pontual sobre sentimento — roteamento dinâmico

Mesmo padrão, agora roteado para o `SentimentAgent`.

In [4]:
show_trace("Qual o sentimento das notícias sobre BBAS3?")

[tool call] get_sentiment_features({'ticker': 'BBAS3'})
[tool result] {"sentiment_score": 0.2584, "sentiment_label": "NEGATIVO", "news_count": 3, "top_headlines": ["Banco pagará R$ 1,1 por ação em juros sobre o capital próprio; veja como aproveitar — Banco ABC aprova R$ 300,9 milhões em juros sobre capital próprio e aumento de capital, enquanto enfr...", "B3 (B3SA3) ajusta valor de JCP e acionistas receberão até R$ 0,22 por ação em julho — A B3 (B3SA3) informou nesta quarta-feira (24) um ajuste nos valores dos juros sobre capital próprio...", "BB Seguridade (BBSE3) pagará R$ 3,8 bilhões em dividendos — A BB Seguridade (BBSE3) aprovou o pagamento de R$ 3,8 bilhões em dividendos, mostra documento enviad..."]}

--- Resposta final ---
{'sentiment_score': 0.2584, 'sentiment_label': 'NEGATIVO', 'news_count': 3, 'top_headlines': ['Banco pagará R$ 1,1 por ação em juros sobre o capital próprio; veja como aproveitar — Banco ABC aprova R$ 300,9 milhões em juros sobre capital próprio e aumento de 

## 4. Pergunta comparativa entre dois tickers

Pergunta envolvendo dois tickers ao mesmo tempo, ainda roteada para o `MarketAgent` — ele decide chamar a tool para cada um dos dois.

In [5]:
show_trace("Compare os indicadores de ITUB4 e BBAS3")

[tool call] get_market_features({'ticker': 'ITUB4'})
[tool call] get_market_features({'ticker': 'BBAS3'})
[tool result] {"close": 40.97, "volume": 22001200, "rsi": 58.14, "macd": 0.0872, "macd_signal_line": -0.2012, "macd_signal": "bullish", "sma_20": 39.71, "sma_50": 41.12, "ema_20": 40.11, "bollinger_upper": 41.43, "bollinger_lower": 38.0}
[tool result] {"close": 19.73, "volume": 16270600, "rsi": 44.54, "macd": -0.3876, "macd_signal_line": -0.5323, "macd_signal": "bullish", "sma_20": 19.62, "sma_50": 21.07, "ema_20": 19.79, "bollinger_upper": 20.58, "bollinger_lower": 18.67}

--- Resposta final ---
ITUB4:
- Preço atual: 40.97
- Volume: 22.001.200
- RSI: 58.14
- MACD: 0.0872
- Linha de sinal MACD: -0.2012
- Sinal MACD: bullish
- SMA 20: 39.71
- SMA 50: 41.12
- EMA 20: 40.11
- Bollinger superior: 41.43
- Bollinger inferior: 38.0

BBAS3:
- Preço atual: 19.73
- Volume: 16.270.600
- RSI: 44.54
- MACD: -0.3876
- Linha de sinal MACD: -0.5323
- Sinal MACD: bullish
- SMA 20: 19.62
- SMA 50: 2

## 5. Histórico de recomendações persistido

Cada recomendação completa gerada (célula 1, e qualquer execução anterior) fica registrada em `data/recommendations.csv`, com todos os indicadores que motivaram a decisão — a base da auditabilidade do sistema.

In [6]:
import pandas as pd

history = load_recommendations()
pd.DataFrame(history).tail(10)

,ticker,date,recommendation,confidence,rsi,macd_signal,sentiment_score,sentiment_label,top_headlines,reasoning
0,PETR4,2026-06-24,AGUARDAR,0.5,28.93,bearish,0.0431,NEGATIVO,['Ibovespa recua pressionado por Petrobras (PE...,RSI em 28.9 (sobrevenda). MACD bearish. Sentim...


## 6. Backtest histórico (Etapas 6 e 6.5)

O agente acima responde com dados do presente. Para saber se a lógica de decisão (`apply_decision_rules`) teria funcionado no passado, o backtest roda a mesma lógica, sem LLM, em cada dia útil dos últimos ~90 dias, comparando contra o retorno real e o Ibovespa. Comparação completa entre sentimento placeholder e sentimento histórico real (GDELT/BigQuery), com gráficos: [`04_backtest_and_evaluation.ipynb`](04_backtest_and_evaluation.ipynb).

In [7]:
from pathlib import Path

from quantumfinance.backtesting.metrics import calculate_metrics

placeholder_path = Path("data/backtest_results.csv")
gdelt_path = Path("data/backtest_results_gdelt.csv")

if placeholder_path.exists() and gdelt_path.exists():
    metrics_placeholder = calculate_metrics(pd.read_csv(placeholder_path))
    metrics_gdelt = calculate_metrics(pd.read_csv(gdelt_path))
    comparison = pd.DataFrame(
        {"placeholder (neutro)": metrics_placeholder, "gdelt (BigQuery)": metrics_gdelt}
    )
else:
    comparison = None
    print("Rode notebooks/04_backtest_and_evaluation.ipynb primeiro para gerar os CSVs do backtest.")

comparison

,placeholder (neutro),gdelt (BigQuery)
accuracy,0.4286,0.4241
beat_ibov_rate,0.5325,0.5312
avg_return_on_buy,-0.526,-0.5882
avg_return_on_sell,-0.6723,-0.7514
recommendation_counts,"{'AGUARDAR': 116, 'COMPRAR': 80, 'VENDER': 55}","{'AGUARDAR': 109, 'COMPRAR': 82, 'VENDER': 53}"
sharpe_proxy,-0.1776,-0.2006


## Limitações e próximos passos

Resumo rápido — detalhes completos no `README.md` e em `docs/decisions.md` / `docs/progress.md`:

- Sentimento em tempo real depende da cobertura dos feeds RSS no momento da consulta — pode cair no fallback neutro sem aviso explícito.
- O backtest usa sentimento histórico via GDELT/BigQuery (score geral de mídia), diferente do FinBERT-PT-BR (especializado em finanças) usado no agente em tempo real.
- Próximos passos: batch queries no BigQuery e download bulk no yfinance, integração com dados da CVM e com o Boletim Focus do Banco Central, e um Context Router Agent para decidir dinamicamente quais esferas temáticas pesquisar por ticker.

Ver [`README.md`](../README.md) para a visão completa do projeto.